# 01 - Data Acquisition
Liest lokale Rohdaten aus data/raw/ - kein Download.

| Datei | Quelle | Inhalt |
|-------|--------|--------|
| pharmacies_ch.geojson | OpenStreetMap | 1640 Apotheken |
| gemeinden_ch_lv95.geojson | swisstopo 2026 | 2123 Gemeinden |
| plz_ch_lv95.geojson | swisstopo | 4073 PLZ |
| stadtquartiere_zuerich.geojson | Stadt Zuerich OGD | 34 Quartiere ZH |

**Projektion:** LV95 (EPSG:2056)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import requests
import io
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)
print('RAW:      ', RAW.resolve())
print('PROCESSED:', PROCESSED.resolve())

In [ ]:
# 1. Apotheken
gdf_pharmacies = gpd.read_file(RAW / 'pharmacies_ch.geojson')
if gdf_pharmacies.crs.to_epsg() != 2056:
    gdf_pharmacies = gdf_pharmacies.to_crs('EPSG:2056')
print('Apotheken:', len(gdf_pharmacies))
print('CRS:      ', gdf_pharmacies.crs)
gdf_pharmacies.head()

In [ ]:
# 2. Gemeinden (swisstopo swissBOUNDARIES3D 2026)
gdf_gemeinden = gpd.read_file(RAW / 'gemeinden_ch_lv95.geojson')
print('Gemeinden:       ', len(gdf_gemeinden))
print('CRS:             ', gdf_gemeinden.crs)
print('Einwohner total: ', gdf_gemeinden['einwohnerzahl'].sum())
gdf_gemeinden[['name','bfs_nummer','einwohnerzahl','gem_flaeche','kantonsnummer']].head()

In [ ]:
# 3. PLZ-Gebiete
gdf_plz = gpd.read_file(RAW / 'plz_ch_lv95.geojson')
if gdf_plz.crs.to_epsg() != 2056:
    gdf_plz = gdf_plz.to_crs('EPSG:2056')
print('PLZ-Gebiete:', len(gdf_plz))
print('CRS:        ', gdf_plz.crs)
gdf_plz.head()

In [ ]:
# 4. Stadtquartiere Zuerich
# Quelle: Stadt Zuerich OGD - Statistische Quartiere WFS
ZH_URL = (
    'https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere'
    '?service=WFS&version=1.1.0&request=GetFeature'
    '&typename=adm_statistische_quartiere_map&outputFormat=GeoJSON'
)
r = requests.get(ZH_URL, timeout=60)
print('Status:', r.status_code)
gdf_quartiere = gpd.read_file(io.BytesIO(r.content))
if gdf_quartiere.crs and gdf_quartiere.crs.to_epsg() != 2056:
    gdf_quartiere = gdf_quartiere.to_crs('EPSG:2056')
gdf_quartiere.to_file(RAW / 'stadtquartiere_zuerich.geojson', driver='GeoJSON')
print('Stadtquartiere ZH:', len(gdf_quartiere))
print('Spalten:', gdf_quartiere.columns.tolist())
gdf_quartiere.head()

In [ ]:
# 5. Uebersichtskarte
gdf_kantone = gpd.read_file(RAW / 'kantone_ch_lv95.geojson')
fig, ax = plt.subplots(figsize=(14, 9))
gdf_kantone.plot(ax=ax, color='#f0f0f0', edgecolor='#999999', linewidth=0.8)
gdf_gemeinden.plot(ax=ax, color='none', edgecolor='#cccccc', linewidth=0.2)
gdf_pharmacies.plot(ax=ax, color='#e63946', markersize=3, alpha=0.7, label='Apotheke')
ax.set_title('Apotheken-Standorte Schweiz (OSM 2024) - 1640 Standorte', fontsize=14, fontweight='bold')
ax.set_axis_off()
ax.legend(loc='lower right')
plt.tight_layout()
out = Path('../outputs/maps/01_apotheken_uebersicht.png')
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Karte gespeichert:', out)

In [ ]:
# 6. Processed speichern
gdf_pharmacies[['osm_id','name','postcode','city','lat','lon','geometry']].to_file(PROCESSED / 'pharmacies_ch.geojson', driver='GeoJSON')
gdf_gemeinden[['bfs_nummer','name','einwohnerzahl','gem_flaeche','kantonsnummer','geometry']].to_file(PROCESSED / 'gemeinden_ch.geojson', driver='GeoJSON')
gdf_plz.to_file(PROCESSED / 'plz_ch.geojson', driver='GeoJSON')
gdf_quartiere.to_file(PROCESSED / 'stadtquartiere_zuerich.geojson', driver='GeoJSON')
for fname in ['pharmacies_ch.geojson','gemeinden_ch.geojson','plz_ch.geojson','stadtquartiere_zuerich.geojson']:
    p = PROCESSED / fname
    size = str(round(p.stat().st_size/1024)) + ' KB' if p.exists() else 'FEHLT'
    print(f'{fname:<40} {size}')
print('Weiter mit: 02_postgis_analysis.ipynb')